In [14]:
import pandas as pd

# Пример загрузки данных из CSV-файла в DataFrame

try:

    sample_data = pd.read_csv('/content/sample_data/california_housing_train.csv')
    print("Пример данных успешно загружен. Головная часть DataFrame:")
    print(sample_data.head())

    # Для демонстрации, если бы в sample_data был 'mental_wellness_index_0_100',
    # мы бы добавили его и вызвали функцию:
    # sample_data['mental_wellness_index_0_100'] = (sample_data['median_house_value'] / 1000).astype(int) % 101 # Пример создания столбца
    # classification_training(sample_data)

    print("\Чтобы использовать функцию `classification_training`, загрузите ваш реальный датасет")
    print("в `pandas.DataFrame` (например, с помощью `pd.read_csv('путь_к_вашему_файлу.csv')`)")
    print("и убедитесь, что он содержит столбец 'mental_wellness_index_0_100' и другие числовые признаки.")
    print("Затем вызовите функцию, передав ей ваш DataFrame: `classification_training(ваш_датафрейм)`")

except FileNotFoundError:
    print("Пример файла '/content/sample_data/california_housing_train.csv' не найден. Убедитесь, что вы работаете в Colab.")
    print("Пожалуйста, загрузите свой файл в DataFrame и вызовите `classification_training`.")
except Exception as e:
    print(f"Произошла ошибка при загрузке или обработке данных: {e}")

# После того как вы загрузите свой DataFrame (например, my_dataframe),
# вызовите функцию следующим образом:
# classification_training(my_dataframe)

Пример данных успешно загружен. Головная часть DataFrame:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -114.31     34.19                15.0       5612.0          1283.0   
1    -114.47     34.40                19.0       7650.0          1901.0   
2    -114.56     33.69                17.0        720.0           174.0   
3    -114.57     33.64                14.0       1501.0           337.0   
4    -114.57     33.57                20.0       1454.0           326.0   

   population  households  median_income  median_house_value  
0      1015.0       472.0         1.4936             66900.0  
1      1129.0       463.0         1.8200             80100.0  
2       333.0       117.0         1.6509             85700.0  
3       515.0       226.0         3.1917             73400.0  
4       624.0       262.0         1.9250             65500.0  
\Чтобы использовать функцию `classification_training`, загрузите ваш реальный датасет
в `pandas.DataFrame` (напри

In [15]:
import pandas as pd
from google.colab import files
import io

print("Пожалуйста, загрузите ваш CSV-файл:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'Загружен файл: {fn}')
    # Предполагаем, что загружен один CSV-файл
    # Вы можете изменить 'my_dataframe' на желаемое имя переменной
    try:
        my_dataframe = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
        print("Файл успешно загружен в DataFrame. Первые 5 строк:")
        print(my_dataframe.head())

        # Теперь вы можете вызвать вашу функцию classification_training:
        # Убедитесь, что ваш DataFrame содержит столбец 'mental_wellness_index_0_100'.
        # Пример добавления столбца, если его нет (для демонстрации):
        # ВНИМАНИЕ: Этот пример создает столбец 'mental_wellness_index_0_100' на основе 'stress_level_0_10' и 'sleep_quality_1_5'.
        # Вам может понадобиться адаптировать эту логику под ваши данные для более осмысленного таргета.
        if 'mental_wellness_index_0_100' not in my_dataframe.columns:
            # Пример: Индекс благополучия как обратная зависимость от стресса и качества сна
            # Предположим, что более низкий стресс и более высокое качество сна указывают на лучшее благополучие.
            # Это простая эвристика, которую вы можете настроить.
            my_dataframe['mental_wellness_index_0_100'] = (
                (10 - my_dataframe['stress_level_0_10']) * 5 +
                my_dataframe['sleep_quality_1_5'] * 10
            ).clip(0, 100).astype(int)
            print("\nСтолбец 'mental_wellness_index_0_100' создан (пример). Первые 5 значений:")
            print(my_dataframe['mental_wellness_index_0_100'].head())

        classification_training(my_dataframe)
        print("\nФункция `classification_training` вызвана.")

    except Exception as e:
        print(f"Ошибка при чтении файла {fn} как CSV: {e}")
        print("Убедитесь, что вы загрузили корректный CSV-файл.")

Пожалуйста, загрузите ваш CSV-файл:


Saving processed_data.csv to processed_data (3).csv
Загружен файл: processed_data (3).csv
Файл успешно загружен в DataFrame. Первые 5 строк:
   age  screen_time_hours  work_screen_hours  leisure_screen_hours  \
0   33              10.79               5.44                  5.35   
1   28               7.40               0.37                  7.03   
2   35               9.78               1.09                  8.69   
3   42              11.13               0.56                 10.57   
4   28              13.22               4.09                  9.13   

   sleep_hours  sleep_quality_1_5  stress_level_0_10  productivity_0_100  \
0         6.63                1.0                9.3                44.7   
1         8.05                3.0                5.7                78.0   
2         6.48                1.0                9.1                51.8   
3         6.89                1.0               10.0                37.0   
4         5.79                1.0               10.0      

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

def classification_training(data: pd.DataFrame):
    """
    Performs classification training on the input DataFrame.

    Args:
        data (pd.DataFrame): The input DataFrame containing 'mental_wellness_index_0_100' and other features.
    """
    # 1. Трансформировать столбец 'mental_wellness_index_0_100' в бинарную целевую переменную
    data['binary_mental_wellness'] = (data['mental_wellness_index_0_100'] >= 15).astype(int)

    # Определяем признаки (X) и целевую переменную (y)
    # Исключаем исходный столбец и новосозданный бинарный, если они не являются признаками
    # Предполагаем, что остальные столбцы являются признаками, если нет других указаний.
    # Для простоты примера выберем несколько столбцов, которые могут быть признаками.
    # В реальном сценарии, 'data' должен содержать только подходящие для моделирования признаки.
    # Здесь используется упрощенный подход, чтобы продемонстрировать функциональность.

    # Пример: если 'mental_wellness_index_0_100' это единственный столбец, который не является признаком
    # X = data.drop(columns=['mental_wellness_index_0_100', 'binary_mental_wellness'])
    # y = data['binary_mental_wellness']

    # Для демонстрации, создадим dummy features, если в данных их недостаточно
    # В реальном использовании, у вас должен быть очищенный датасет с фичами.
    # Если `data` не содержит других числовых колонок кроме 'mental_wellness_index_0_100',
    # то нужно будет создать или выбрать подходящие признаки.
    # Допустим, что `data` имеет как минимум один другой числовой столбец для X.

    # **ВАЖНО**: Замените `['feature1', 'feature2']` на актуальные столбцы признаков из вашего DataFrame.
    # Если у вас много столбцов, возможно, потребуется более интеллектуальный способ их выбора.

    # Пример выбора всех числовых столбцов, кроме целевых
    numeric_cols = data.select_dtypes(include=['number']).columns
    feature_cols = [col for col in numeric_cols if col not in ['mental_wellness_index_0_100', 'binary_mental_wellness']]

    if not feature_cols:
        raise ValueError("DataFrame does not contain enough numeric features for training. Please provide a DataFrame with suitable features.")

    X = data[feature_cols]
    y = data['binary_mental_wellness']

    # Разделение данных на тренировочную и тестовую выборки
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

    # 2. Создавать словарь-сетку для поиска оптимального набора гиперпараметров
    param_grid = {
        'n_estimators': [50, 100, 200],  # Количество деревьев в лесу
        'max_depth': [None, 10, 20],  # Максимальная глубина дерева
        'min_samples_split': [2, 5],  # Минимальное количество выборок, необходимое для разделения внутреннего узла
        'min_samples_leaf': [1, 2]  # Минимальное количество выборок, необходимое для листового узла
    }

    # 3. Создавать экземпляр класса случайного леса
    rf = RandomForestClassifier(random_state=42)

    # Инициализация GridSearchCV
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

    # Обучение GridSearchCV
    grid_search.fit(X_train, y_train)

    # 4. Выполнять поиск оптимального набора гиперпараметров и выводить в консоль этот набор
    print(f"Оптимальные гиперпараметры: {grid_search.best_params_}")

    # 5. Обучать финальную модель с оптимальным набором гиперпараметров
    best_rf_model = grid_search.best_estimator_

    # Прогнозирование на тестовой выборке
    y_pred = best_rf_model.predict(X_test)

    # 6. Выводить в консоль значения метрик: accuracy и f1 score
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"RF: {accuracy:.4f}; {f1:.4f}")